# Pydantic — Quick Tutorial

**Pydantic** is a Python library for **data validation** using regular Python type hints.

When data comes into your program (from a JSON request, a config file, an LLM response, etc.) you usually want to:

1. Make sure it has the right shape (correct fields, correct types).
2. Reject or fix it if it does not.
3. Work with it as a normal Python object.

Pydantic does all of this automatically — you just describe the shape of your data with a class.

It is used by **FastAPI**, **LangChain**, the **OpenAI SDK**, and most modern LLM frameworks for structured outputs.

## Installation

Install with pip:

```bash
pip install pydantic
```

If you also want settings management from `.env` files (covered later):

```bash
pip install pydantic-settings
```

In [ ]:
# Check that Pydantic is installed and which version is being used
import pydantic
print("Pydantic version:", pydantic.VERSION)

## What is `BaseModel`?

`BaseModel` is the **main building block** of Pydantic. You inherit from it to create your own data models.

Think of it as a smarter version of a regular Python class:

- You declare each field with a **name** and a **type hint** (e.g. `id: int`).
- When you create an instance, Pydantic checks that every value matches its declared type.
- If a value can be safely converted (e.g. the string `"42"` to the int `42`), it does that for you.
- If a value is wrong, you get a clear `ValidationError` instead of a silent bug.
- The instance also gets useful methods like `.model_dump()` (to a dict) and `.model_dump_json()` (to JSON).

In short: **`BaseModel` turns a class definition into a self-validating data container.**

## 1. Define a Model

Create a subclass of `BaseModel` and list the fields with their types. `is_active: bool = True` shows how to give a field a default value.

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    is_active: bool = True   # default value — optional when creating a User

user = User(id=1, name="Pradeep")
print(user)
print("name:", user.name)
print("is_active:", user.is_active)

## 2. Validation Errors

If a value cannot be converted to the declared type, Pydantic raises a `ValidationError` that tells you exactly which field failed and why. This is what makes Pydantic so useful — bad data fails fast and loud, not silently.

In [ ]:
from pydantic import ValidationError

try:
    User(id="abc", name="Bob")   # "abc" cannot become an int
except ValidationError as e:
    print(e)

## 3. Field Constraints

Sometimes a type alone is not enough — you also want rules like *price must be greater than 0* or *quantity cannot be negative*. The `Field` helper lets you attach these constraints (and defaults, and descriptions) to any field.

In [ ]:
from pydantic import Field

class Product(BaseModel):
    name: str
    price: float = Field(gt=0)            # gt = greater than
    qty: int = Field(ge=0, default=0)     # ge = greater than or equal

p = Product(name="Pen", price=10, qty=5)
print(p)

## 4. Nested Models

A field's type can itself be another `BaseModel`. Pydantic will validate the inner object too — even if you pass a plain dict, it will be converted into the nested model automatically.

In [ ]:
class Address(BaseModel):
    city: str
    zip: str

class Customer(BaseModel):
    name: str
    address: Address   # nested model

c = Customer(name="Alice", address={"city": "Bangalore", "zip": "560001"})
print(c)
print("city:", c.address.city)

## 5. Convert to / from JSON

Pydantic models have built-in helpers to move between Python objects, dicts, and JSON strings — perfect for APIs and LLM responses.

- `model_dump()` → Python `dict`
- `model_dump_json()` → JSON string
- `model_validate_json(...)` → parse a JSON string back into a model

In [ ]:
user = User(id=1, name="Pradeep")

print("as dict:", user.model_dump())
print("as json:", user.model_dump_json())

# Parse JSON back into a User object
u2 = User.model_validate_json('{"id": 2, "name": "Bob"}')
print("reloaded:", u2)

## Summary

- `BaseModel` is a class you inherit from to define a validated data shape.
- Type hints describe each field; Pydantic enforces them at runtime.
- `Field` adds extra rules (min/max, defaults, descriptions).
- Models can be nested.
- `model_dump()` / `model_dump_json()` / `model_validate_json()` handle conversion.

This same pattern powers **FastAPI** request bodies, **LangChain** structured outputs, and tool/function calling with LLMs.